# Fase 3: Explicabilidad (XAI) con SHAP

El objetivo de este notebook es desentrañar el modelo XGBoost ('caja negra') para entender **por qué** decide que un motor va a fallar, tanto de forma global (importancia general de sensores) como local (análisis de un motor específico).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

# Inicializamos JS de SHAP para que se integre visualmente en el notebook
shap.initjs()

### 1. Carga de Datos y del Modelo
Cargamos la configuración del split para extraer exactamente la misma matriz de datos (Validación) que usó el predictor en `train.py`.

In [ ]:
from src.data import load_cmapss_data, get_train_val_split
from src.features import add_binary_target, drop_constant_columns, add_rolling_features

# 1.1 Cargar y procesar datos idéntico a train.py
df = load_cmapss_data('train_FD001.txt')
df = add_binary_target(df, window_size=30)
df = drop_constant_columns(df)
df = add_rolling_features(df, windows=[5, 15])
df.fillna(0, inplace=True)

# 1.2 Mismo Split de Entrenamiento y Validación
train_df, val_df = get_train_val_split(df, val_size=0.2)
target_col = 'failure_within_30_cycles'
cols_to_drop = ['unit_id', 'rul', target_col]
X_train = train_df.drop(columns=cols_to_drop)
y_train = train_df[target_col]
X_val = val_df.drop(columns=cols_to_drop)
y_val = val_df[target_col]

# 1.3 Cargar Modelo Entrenado y Guardado
model = joblib.load('../models/xgboost_baseline.joblib')
print("Modelo cargado correctamente.")

### 2. Inicialización del Explainer de SHAP
Al tratarse de XGBoost (ensamblador basado en árboles), usamos el motor ultra-rápido `TreeExplainer` de SHAP.

In [ ]:
explainer = shap.TreeExplainer(model)

# Calculamos los valores SHAP sobre nuestro set de Validación ciego
shap_values = explainer.shap_values(X_val)

### 3. Explicabilidad Global (Buscando los Factores Clave)
**¿Qué variables dominan la decisión del motor predictivo globalmente?**

- Cada punto del gráfico corresponde a **un ciclo individual de un motor**.
- **Color Rojo:** Valor alto de ese sensor.
- **Color Azul:** Valor bajo de ese sensor.
- **Eje Horizontal (SHAP value):** Impacto sobre la meta 'Fallo Inminente'. A mayor valor a la derecha, más convencido está el modelo de que va a romperse.

In [ ]:
# Visualizar la dispersión e impacto direccional
shap.summary_plot(shap_values, X_val, plot_type="dot", max_display=12)

### 4. Importancia Base (Modo Clásico)
Por si necesitamos un gráfico tradicional para una presentación de negocio (promedio absoluto del impacto).

In [ ]:
shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=12)

### 5. Explicabilidad Local (Inspección por Turbina)
Supongamos que en la pantalla de control la alarma se dispara al máximo nivel. ¿Qué valores **exactos** han detonado el aviso de emergencia?

In [ ]:
# Predecimos la probabilidad cruda (0 a 1) para cada instancia en Validación
y_prob = model.predict_proba(X_val)[:, 1]

# Encontremos la fila exacta de Validación donde la curva de probabilidad indica un riesgo extremo de fallo inminente.
idx_peligroso = y_prob.argmax() 

print(f"Analizando el ciclo en el índice {idx_peligroso} con un nivel de criticidad del {y_prob[idx_peligroso]*100:.2f}% de probabilidad de explotar...\n")

# Gráfico Waterfall para ver exactamente cómo cada sensor ha empujado el aviso matemático final hacia el riesgo.
shap.waterfall_plot(shap.Explanation(values=shap_values[idx_peligroso],
                                      base_values=explainer.expected_value,
                                      data=X_val.iloc[idx_peligroso],
                                      feature_names=X_val.columns))